## 1.7 Güvenli Mesaj İletimi (Secure message transmission)

Ele alacağımız ilk kriptografik problem, klasik ve kuantum şifrelemenin temel taşı olan güvenli mesaj iletimidir. Bu senaryoda, başrollerimiz dürüst taraflar olan Alice ve Bob'dur. Alice ve Bob, aralarındaki iletişimi meraklı bir casus olan Eve'in (saldırgan/eavesdropper) meraklı gözlerinden korumak istemektedirler. 

Bu modellerde Alice ve Bob her zaman dürüsttür. Kendi laboratuvarları üzerinde tam kontrol sahibidirler; yani Eve bu laboratuvarların içine asla sızamaz veya gizlice gözetleyemez. Ancak Eve, Alice ve Bob'u birbirine bağlayan halka açık iletişim kanalına tam erişim hakkına sahiptir; kanaldan geçen her türlü veriyi dinleyebilir ve kopyalayabilir.

Alice ve Bob'un mesajlarını güvenli bir şekilde iletmek için kullanabilecekleri en temel (ve matematiksel olarak en güvenli) yöntem, mesajı kodlamak için gizli bir **anahtar (key)** kullanmaktır. Bu anahtarın hem Alice hem de Bob tarafından bilindiği, ancak tamamen kendilerine özel olduğu varsayılır: Casus Eve'in bu anahtar hakkında kesinlikle hiçbir bilgisi yoktur. Bu nedenle, bu tür sistemlere **gizli anahtarlı kriptosistemler (private-key cryptosystems)** adı verilir. 



### 1.7.1 Shannon’ın Gizlilik Şartı ve Büyük Anahtar İhtiyacı (Shannon’s secrecy condition and the need for large keys)

Alice ve Bob'un, casus Eve tarafından bilinmeyen klasik bir $k$ anahtarını paylaştığını varsayalım. Bu "bilinmeme" durumunu ilerleyen adımlarda matematiksel olarak daha kesin hale getireceğiz. Şimdilik sezgisel bir tanım yapacak olursak: Eğer Eve'in elindeki veriler ile bu anahtar arasında hiçbir ilişki (korelasyon) yoksa, Eve anahtarı kesinlikle bilmiyor demektir. Bu durumda, olası tüm $|K|$ adet anahtar için her bir anahtarın seçilme olasılığı birbirine eşittir:

$$p(k) = \frac{1}{|K|} \quad \text{--- (1.59)}$$

Gizli mesajların güvenli bir şekilde aktarılmasına ilişkin matematiksel çerçeve ilk olarak Claude Shannon tarafından ortaya konmuştur [Sha49]. Herhangi bir şifreleme şeması (encryption scheme) temelde iki ana fonksiyondan oluşur:

1. **Şifreleme Fonksiyonu ($\text{Enc}$):** Gizli anahtarı ($k$) ve orijinal mesajı ($m$ - düz metin / plaintext) girdi olarak alan ve bunu şifreli bir mesaja ($e$ - şifreli metin / ciphertext) eşleyen doğrusal bir fonksiyondur:
   $$\text{Enc}(k, m) = e \quad \text{--- (1.60)}$$

2. **Deşifreleme Fonksiyonu ($\text{Dec}$):** Gizli anahtarı ($k$) ve şifreli metni ($e$) girdi olarak alıp, sistemi yeniden orijinal düz metne ($m$) döndüren fonksiyondur:
   $$\text{Dec}(k, e) = m \quad \text{--- (1.61)}$$


In [1]:
import secrets

def bytes_to_bits(b_data):
    """Metni bit dizisine dönüştürür."""
    return "".join(f"{b:08b}" for b in b_data)

def bits_to_bytes(bit_str):
    """Bit dizisini tekrar metne dönüştürür."""
    by_arr = bytearray()
    for i in range(0, len(bit_str), 8):
        by_arr.append(int(bit_str[i:i+8], 2))
    return bytes(by_arr)

# 1. Giriş: Alice'in göndermek istediği orijinal düz metin (m)
mesaj = "KUANTUM"
mesaj_bytes = mesaj.encode('utf-8')
m_bits = bytes_to_bits(mesaj_bytes)
N = len(m_bits)

# 2. Anahtar Üretimi: Shannon şartı gereği p(k) = 1/|K| olan tamamen rastgele bir k anahtarı
# secrets modülü kriptografik güvenliğe sahip gerçek rastgele bitler üretir
k_bits = "".join(str(secrets.randbelow(2)) for _ in range(N))

# 3. Şifreleme Fonksiyonu Enc(k, m) = e (Bit düzeyinde XOR işlemi)
e_bits = "".join(str(int(m_bits[i]) ^ int(k_bits[i])) for i in range(N))

# 4. Deşifreleme Fonksiyonu Dec(k, e) = m (Yeniden XOR işlemiyle orijinal metne dönüş)
m_cozulen_bits = "".join(str(int(e_bits[i]) ^ int(k_bits[i])) for i in range(N))
mesaj_cozulen = bits_to_bytes(m_cozulen_bits).decode('utf-8')

print(f"Alice'in Orijinal Mesajı (m): {mesaj}")
print(f"Kriptografik Rastgele Anahtar (k): {k_bits[:20]}...")
print(f"Kanaldan Geçen Şifreli Metin (e) [Eve'in gördüğü]: {e_bits[:20]}...")
print(f"Bob'un Deşifre Ettiği Mesaj (Dec): {mesaj_cozulen}")


Alice'in Orijinal Mesajı (m): KUANTUM
Kriptografik Rastgele Anahtar (k): 11100111110101110100...
Kanaldan Geçen Şifreli Metin (e) [Eve'in gördüğü]: 10101100100000100000...
Bob'un Deşifre Ettiği Mesaj (Dec): KUANTUM


> 🟦 **Tanım 1.7.1 — Mükemmel Gizlilik / Güvenlik.** Bir $(\text{Enc}, \text{Dec})$ şifreleme şeması, ancak ve ancak mesajlar üzerindeki tüm önsel (prior) $p(m)$ olasılık dağılımları ve tüm olası $m$ mesajları için aşağıdaki koşul sağlanıyorsa **gizli (secret)** veya **mükemmel güvenli (perfectly secure)** kabul edilir:
> 
> $$p(m) = p(m \mid e) \quad \text{--- (1.64)}$$
> 
> Burada $e = \text{Enc}(k,m)$ ifadesi, mesajın gizli bir $k$ anahtarı ile şifrelenmiş halidir.

Başka bir deyişle; kanaldan geçen $e$ şifreli metnini gizlice dinleyen veya ele geçiren bir casus (Eve), bu şifreli metne sahip olduktan sonra, orijinal mesajın içeriği ($m$) hakkında şifreli metni **hiç görmediği duruma kıyasla** sıfır (0) ek bilgi kazanmalıdır. 

Yani, dışarıdaki herhangi bir insanın mesajı rastgele tahmin etme olasılığı olan önsel $p(m)$ değeri ile, elinde şifreli metin ($e$) bulunan Eve'in gözünden o mesajın gelme koşullu olasılığı $p(m \mid e)$ kuruşu kuruşuna tamamen aynı kalmalıdır. Bu, kriptografide tanımlanabilecek en katı ve en güçlü güvenlik ölçütüdür: Şifreli metne ($e$) erişim sağlamak, casusa mutlak surette **hiçbir bilgi kazandırmaz!**


Yalnızca yukarıdaki gizlilik şartını sağlayan bir şifreleme şeması bulmanın aslında "aşırı kolay" olduğunu fark etmek önemlidir: Örneğin Alice, Bob'a orijinal mesajla tamamen bağımsız, rastgele seçilmiş bir $e$ dizesi gönderebilir. Bu senaryoda Eve şifreli metinden mesaj hakkında hiçbir şey öğrenemez; yani mükemmel gizlilik şartı ($p(m|e) = p(m)$) teknik olarak sağlanır.

Ancak bu noktada, haklı olarak böyle bir sistemin tamamen kullanışsız ve anlamsız olduğunu öne sürerek itiraz edeceksinizdir! Eğer gönderilen $e$ dizesinin orijinal $m$ mesajıyla hiçbir bağı veya ilişkisi yoksa, Bob bu anlamsız dizeden orijinal mesajı nasıl geri öğrenebilir ki? 

İşte bu nedenle, bir şifreleme şemasının hem güvenli hem de işlevsel olabilmesi için sağlaması gereken ikinci temel koşul, onun **doğru (correct / correctness)** olmasıdır. Yani dürüst alıcı Bob, elindeki doğru gizli anahtarı kullandığında, orijinal mesajı hatasız bir şekilde deşifre edebilmelidir.


> 📝 **Sayısal Örnek:** Alice'in "0" veya "1" bitlerinden birini eşit olasılıkla ($p(0)=0.5, p(1)=0.5$) göndermek istediğini varsayalım. Alice, One-Time Pad yöntemiyle mesajını rastgele bir gizli anahtarla şifreliyor ve kanaldan şifreli metin olarak **$e=1$** geçiyor. 
> 
> Mükemmel gizlilik şartı gereği, Eve kanaldan geçen $e=1$ bilgisini görse bile, orijinal mesajın "0" olma olasılığı hala tam olarak %50 kalmalıdır ($p(0 \mid e=1) = p(0) = 0.5$).


In [2]:
import numpy as np

# Önsel olasılıklar (Mesajın önceden tahmin edilme olasılığı)
p_m0 = 0.5  # p(m=0)
p_m1 = 0.5  # p(m=1)

# One-Time Pad şifreleme altında, anahtar rastgele olduğu için 
# Mesaj ne olursa olsun şifreli metnin e=1 gelme olasılığı tamamen eşittir:
p_e1_given_m0 = 0.5  # p(e=1 | m=0) -> anahtar 1 seçildiyse
p_e1_given_m1 = 0.5  # p(e=1 | m=1) -> anahtar 0 seçildiyse

# Toplam Olasılık Teoremi ile genel p(e=1) olasılığını hesaplayalım
p_e1 = (p_e1_given_m0 * p_m0) + (p_e1_given_m1 * p_m1)

# Bayes Teoremi ile e=1 bilindiğinde mesajın m=0 olma koşullu olasılığı: p(m=0 | e=1)
p_m0_given_e1 = (p_e1_given_m0 * p_m0) / p_e1

print(f"Önsel Olasılık p(m=0): {p_m0}")
print(f"Şifreli Metin (e=1) görüldükten sonraki Sonrasal Olasılık p(m=0 | e=1): {p_m0_given_e1}")
print(f"Shannon Mükemmel Gizlilik Şartı Sağlandı mı?: {np.isclose(p_m0, p_m0_given_e1)}")


Önsel Olasılık p(m=0): 0.5
Şifreli Metin (e=1) görüldükten sonraki Sonrasal Olasılık p(m=0 | e=1): 0.5
Shannon Mükemmel Gizlilik Şartı Sağlandı mı?: True


> 🟦 **Tanım 1.7.2 — Doğruluk (Correctness).** Bir $(\text{Enc}, \text{Dec})$ şifreleme şeması, ancak ve ancak tüm olası $m$ mesajları ve tüm olası $k$ anahtarları için aşağıdaki koşul sağlanıyorsa **doğru (correct)** kabul edilir:
> 
> $$m = \text{Dec}\big(k, \, \text{Enc}(k,m)\big) \quad \text{--- (1.64)}$$

Tıpkı gizlilik şartında olduğu gibi, yalnızca "doğru" olan bir şifreleme şeması bulmak da aşırı kolaydır: Örneğin Alice, şifreli metin olarak doğrudan mesajın kendisini Bob'a gönderebilir ($e = m$). Bob hiçbir işlem yapmadan mesajı okur; yani doğruluk şartı kusursuzca sağlanır. Ancak bu kez de casus Eve kanaldan geçen tüm mesajları doğrudan okuyabilir; bu da tam olarak engellemek istediğimiz şeydir!

Kriptografi sanatı, **hem doğru hem de güvenli** olan protokolleri aynı anda tasarlayabilmektir. Neredeyse tüm senaryolarda tek başına doğruluğu veya tek başına güvenliği sağlamak çok kolayken, asıl büyük zorluk bu iki koşulu tek bir çatı altında birleştirmek istediğimizde ortaya çıkar.


Alice ve Bob'un paylaştığını varsaydığımız gizli anahtar, hem doğru hem de mükemmel gizli bir şifreleme şeması elde etmek için en kritik bileşendir. Peki, böyle büyük bir anahtara gerçekten ihtiyaç var mıdır? 

Yanıtın evet olduğu ortaya çıkıyor: Sadece ihtiyaç duyulmakla kalmaz, aslında olası mesaj sayısı kadar farklı anahtara sahip olmak zorundayızdır. (Bir mesajın olası kabul edilmesi için önsel olasılığının $p(m) > 0$ olması gerekir). Bu gerçeği, Shannon'a dayanan şu lemma ile resmileştirelim:

> 📜 **Lemma 1 — Shannon'ın Anahtar Sınırı.** Bir $(\text{Enc}, \text{Dec})$ şifreleme şeması, ancak ve ancak olası anahtarların sayısı ($|K|$), olası mesajların sayısından ($|M|$) az değilse; yani **$|K| \geq |M|$** şartı sağlanıyorsa aynı anda hem doğru hem de güvenli olabilir.


**İspat:**
Aksini varsayalım (olmayana ergi / çelişki yöntemi): Elimizde daha az anahtar kullanan, yani $|K| < |M|$ şartına sahip mükemmel güvenli bir şemamız olsun. Bu tür bir sistemin aslında güvenli olamayacağını gösterelim. Kanaldan geçen $e$ şifreli metnini ele geçiren bir casus Eve'i düşünelim. Eve, elindeki şifreli metni ($e$) olası tüm $k$ anahtarlarıyla tek tek deşifre etmeyi denerse, şu olası mesajlar kümesini ($\mathcal{S}$) hesaplayabilir:

$$\mathcal{S} = \{ \hat{m} \mid \exists k, \, \hat{m} = \text{Dec}(k,e) \} \quad \text{--- (1.65)}$$

Yani $\mathcal{S}$, gözlemlenen $e$ şifreli metnini üretebilecek tüm potansiyel $\hat{m}$ mesajlarının kümesidir. Olası her bir $k$ anahtarı için en fazla bir adet mesaj elde edebileceğimizden, bu kümenin boyutu en fazla anahtar sayısı kadar olabilir: $|\mathcal{S}| \leq |K|$. En baştaki çelişki varsayımımız gereği $|K| < |M|$ olduğundan, şu kesin eşitsizliğe ulaşırız:

$$|\mathcal{S}| < |M|$$

Bu durum, kümenin dışında kalan ($m \notin \mathcal{S}$) en az bir adet olası $m$ mesajının mutlaka var olduğu anlamına gelir. Eğer mesaj $\mathcal{S}$ kümesinin dışındaysa, casus Eve şu sonuca varır: Kanaldan geçen $e$ şifreli metnini üretebilecek hiçbir gizli anahtar mevcut değildir; o halde bu mesaj kesinlikle gönderilmiş olamaz! Matematiksel olarak bu durum, Eve'in gözünden o mesajın gelme koşullu olasılığını sıfıra indirir:

$$p(m \mid e) = 0$$

Oysa bu mesaj en başta olası bir mesaj olduğu için $p(m) > 0$ şartına sahipti. Sonuç olarak **$0 = p(m \mid e) \neq p(m) > 0$** çelişkisi elde edilir. Bu durum Tanım 1.7.1'deki mükemmel gizlilik şartını tamamen ihlal eder. Dolayısıyla, bir şifreleme şemasının güvenli olabilmesi için anahtar uzayının boyutu mesaj uzayının boyutundan küçük olamaz ($|K| \geq |M|$). 




In [3]:
import numpy as np

# 1. Senaryo: Mesaj uzayımız 3 olası kelimeden oluşsun (A, B, C) -> |M| = 3
mesajlar = ["A", "B", "C"]
# Her mesajın önsel olasılığı eşit ve p(m) > 0
p_m = 1 / len(mesajlar) 

# 2. Hatalı Seçim: Teoremin aksine daha az sayıda anahtar kullanalım -> |K| = 2
anahtarlar = ["k1", "k2"]

# Varsayalım ki Alice "A" mesajını şifreledi ve kanaldan "e_test" şifreli metni geçti.
# Bob'un deşifre fonksiyonunun (Dec) bu 2 anahtarla üretebileceği olası sonuç kümesi (S)
# k1 ile "A" çıksın, k2 ile "B" çıksın.
S_kumesi = ["A", "B"] # |S| <= |K| olduğu için "C" mesajı dışarıda kaldı!

print(f"Olası Tüm Mesaj Uzayı (M): {mesajlar} (Boyut: {len(mesajlar)})")
print(f"Casus Eve'in Şifreli Metni Çözerek Bulduğu Küme (S): {S_kumesi} (Boyut: {len(S_kumesi)})")

# Bilgi Sızıntısı Kontrolü: Kümede olmayan "C" mesajı için casusun çıkarımı
print(f"\nEve'in Analizi: 'C' mesajı S kümesinde olmadığı için p(C | e) = 0.00")
print(f"Oysa önsel olasılık p(C) = {p_m:.2f} idi.")

# Shannon Koşulu Denetimi: p(m|e) == p(m) mi?
gizlilik_saglandi_mi = np.isclose(0.0, p_m)
print(f"\nMükemmel Gizlilik Sağlandı mı?: {gizlilik_saglandi_mi} (Lemma 1 doğrulanmıştır, |K| < |M| şifrelemeyi güvensiz yapar!)")


Olası Tüm Mesaj Uzayı (M): ['A', 'B', 'C'] (Boyut: 3)
Casus Eve'in Şifreli Metni Çözerek Bulduğu Küme (S): ['A', 'B'] (Boyut: 2)

Eve'in Analizi: 'C' mesajı S kümesinde olmadığı için p(C | e) = 0.00
Oysa önsel olasılık p(C) = 0.33 idi.

Mükemmel Gizlilik Sağlandı mı?: False (Lemma 1 doğrulanmıştır, |K| < |M| şifrelemeyi güvensiz yapar!)
